In [3]:
import csv
import pandas as pd

# Load CSV file (all cols as strings
df = pd.read_csv("data/transactions.csv", dtype=str)

col1 = 'sender_account'
col2 = 'receiver_account'
match_value = 'SE8902'

col1_matches = df[col1].dropna().str.startswith(match_value)
col2_matches = df[col2].dropna().str.startswith(match_value)

# Get values that match and concatenate
values_col1 = df.loc[col1_matches, col1]
values_col2 = df.loc[col2_matches, col2]

# Use pd.concat instead of append, then convert to a set
unique_values = set(pd.concat([values_col1, values_col2]))

# Convert the set to a sorted list (optional, for consistent output order)
# unique_list = sorted(unique_values)

# Write to a new CSV file as a single row
#with open("./data/account_nrs.csv", mode="w", newline='', encoding='utf-8') as file:
#   writer = csv.writer(file)
#   writer.writerow(unique_list)  # Writes all values in one row, comma-delimited

# print("Wrote " + str(len(unique_list)) + " unique values")

In [4]:
import random
from faker import Faker

fake = Faker('sv_SE')  # Swedish locale

# Step 1: Generate 800 unique customers with full profile
customer_profiles = []
customer_names = set()

while len(customer_names) < 800:
    name = fake.name()
    if name not in customer_names:
        customer_names.add(name)
        customer_profiles.append({
            "Customer": name,
            "Address": fake.address().replace("\n", ", "),  # clean up multiline addresses
            "Phone": fake.phone_number(),
            "Personnummer": fake.ssn()
        })

# Step 2: Distribute 1000 bank accounts randomly
account_numbers = list(unique_values)  # <- your existing list of 1000 SE8902... strings
random.shuffle(account_numbers)

# Create a mapping: personnummer -> list of accounts
customer_accounts = {profile["Personnummer"]: [] for profile in customer_profiles}

for account in account_numbers:
    chosen = random.choice(customer_profiles)
    customer_accounts[chosen["Personnummer"]].append(account)

# Step 3: Flatten to a list of rows with full profile
rows = []
for profile in customer_profiles:
    accounts = customer_accounts[profile["Personnummer"]]
    for account in accounts:
        rows.append({
            "Customer": profile["Customer"],
            "Address": profile["Address"],
            "Phone": profile["Phone"],
            "Personnummer": profile["Personnummer"],
            "BankAccount": account
        })

# Step 4: Save to CSV
df = pd.DataFrame(rows)
df.to_csv("data/sebank_customers_with_accounts.csv", index=False)

print(f"Wrote {len(rows)} bank account assignments for {len(customer_profiles)} customers.")


Wrote 1000 bank account assignments for 800 customers.
